                # Circuit Quantization

                This notebook documents the graph → symbolic circuit → mode decomposition → numerical quantization workflow exposed by `Circuit` and its supporting APIs.
                

                **Audience:** readers who want the Julia equivalent of the scqubits custom-circuit path.

                **Prerequisites:** familiarity with branch-based superconducting circuit descriptions and symbolic Hamiltonians.

                **Learning goals:** parse a circuit, inspect its topology, build a `SymbolicCircuit`, construct a quantized `Circuit`, inspect symbolic expressions, and sweep flux or branch parameters.
                

                ## Outline

                1. Parse a YAML-style circuit description into a `CircuitGraph`.
                2. Inspect spanning trees, closure branches, and superconducting loops.
                3. Build the symbolic circuit and examine the variable transformation.
                4. Construct a quantized `Circuit`, inspect symbolic accessors, and update parameters.
                

In [ ]:
                using ScQubitsMimic

                desc = """
                branches:
                  - [JJ, 0, 1, EJ=10.0, EC=0.3]
                  - [JJ, 0, 1, EJ=10.0, EC=0.3]
                  - [C, 0, 1, EC=0.5]
                """
                

In [ ]:
                cg = parse_circuit(desc)
                (
                    branch_types = [branch.branch_type for branch in cg.branches],
                    num_nodes = cg.num_nodes,
                    has_ground = cg.has_ground,
                )
                

In [ ]:
                tree = find_spanning_tree(cg)
                closure = find_closure_branches(cg, tree)
                sc_closure, sc_loops = find_superconducting_loops(cg)

                (
                    spanning_tree = tree,
                    closure_branches = closure,
                    superconducting_closure_branches = sc_closure,
                    superconducting_loops = sc_loops,
                    fundamental_loops = find_fundamental_loops(cg, tree),
                )
                

In [ ]:
                sc = build_symbolic_circuit(cg)
                T, categories = compute_variable_transformation(sc)
                (
                    symbolic_fluxes = string.(sc.external_fluxes),
                    symbolic_offset_charges = string.(sc.offset_charges),
                    transform = round.(T, digits=3),
                    categories = categories,
                )
                

In [ ]:
                circ = Circuit(desc; ncut=30)
                (
                    mode_categories = circ.var_categories,
                    cutoffs = circ.cutoffs,
                    symbolic_hamiltonian = sym_hamiltonian(circ; return_expr=true),
                    symbolic_hamiltonian_node = sym_hamiltonian_node(circ),
                    symbolic_lagrangian_node = sym_lagrangian(circ),
                    symbolic_lagrangian_mode = sym_lagrangian(circ; vars_type=:new),
                )
                

In [ ]:
                (
                    transformation = round.(first(variable_transformation(circ)), digits=3),
                    offset_charge_map = offset_charge_transformation(circ),
                    loop_to_flux_map = sym_external_fluxes(circ),
                    symbolic_offset_charge_labels = string.(offset_charges(circ)),
                )
                

In [ ]:
                set_external_flux!(circ, 1, π / 2)
                set_offset_charge!(circ, 1, 0.125)
                before_override = (
                    Φ1 = get_param(circ, Symbol("Φ1")),
                    ng1 = get_param(circ, :ng1),
                    EJ_branch_1 = get_param(circ, :EJ_1),
                )

                set_param!(circ, :EJ_1, 12.0)
                after_override = (
                    Φ1 = get_param(circ, Symbol("Φ1")),
                    ng1 = get_param(circ, :ng1),
                    EJ_branch_1 = get_param(circ, :EJ_1),
                    listed_params = list_branch_params(circ),
                )

                (before_override = before_override, after_override = after_override)
                

In [ ]:
                sd = get_spectrum_vs_paramvals(circ, Symbol("Φ1"), collect(range(0.0, π; length=5)); evals_count=3)
                (
                    Φ1_points = round.(sd.param_vals, digits=4),
                    ω01_vs_flux = round.(sd.eigenvalues[:, 2] .- sd.eigenvalues[:, 1], digits=6),
                )
                

In [ ]:
                invalidate_cache!(circ)
                typeof(hamiltonian(circ))
                

                ## Exercise

                Add a shunt capacitor branch `[C, 0, 1, EC=1.0]` to the circuit description and compare the resulting `ω01` against the unshunted device.
                

In [ ]:
                desc_exercise = """
                branches:
                  - [JJ, 0, 1, EJ=10.0, EC=0.3]
                  - [JJ, 0, 1, EJ=10.0, EC=0.3]
                  - [C, 0, 1, EC=0.5]
                  - [C, 0, 1, EC=1.0]
                """
                circ_exercise = Circuit(desc_exercise; ncut=30)
                round(diff(eigenvals(circ_exercise; evals_count=2))[1], digits=6)
                

                ## Pitfalls And Extensions

                Circuit external flux symbols such as `Φ1` live in radians, not in units of `Φ₀`. This differs from the hardcoded `TunableTransmon` API.

                `invalidate_cache!` clears the numerical Hamiltonian cache. If hierarchical diagonalization has already been configured, the symbolic decomposition remains available while the numerical hierarchy is rebuilt on demand.
                

                ## API Covered

                `BranchType`, `C_branch`, `L_branch`, `JJ_branch`, `CJ_branch`, `Branch`, `CircuitGraph`, `parse_circuit`, `find_spanning_tree`, `find_closure_branches`, `find_fundamental_loops`, `find_superconducting_loops`, `SymbolicCircuit`, `build_symbolic_circuit`, `compute_variable_transformation`, `VarCategories`, `Circuit`, `variable_transformation`, `offset_charge_transformation`, `external_fluxes`, `sym_external_fluxes`, `offset_charges`, `sym_hamiltonian`, `sym_hamiltonian_node`, `sym_lagrangian`, `set_external_flux!`, `set_offset_charge!`, `set_param!`, `get_param`, `list_branch_params`, and `invalidate_cache!`.
                